# Wu

## Index
1. [Instantiate model class](#Instantiate-model-class)
2. [Define clock metadata](#Define-clock-metadata)
3. [Download clock dependencies](#Download-clock-dependencies)
5. [Load features](#Load-features)
6. [Load weights into base model](#Load-weights-into-base-model)
7. [Load reference values](#Load-reference-values)
8. [Load preprocess and postprocess objects](#Load-preprocess-and-postprocess-objects)
10. [Check all clock parameters](#Check-all-clock-parameters)
10. [Basic test](#Basic-test)
11. [Save torch model](#Save-torch-model)
12. [Clear directory](#Clear-directory)

Let's first import some packages:

In [1]:
import os
import inspect
import shutil
import json
import torch
import pandas as pd
import pyaging as pya

## Instantiate model class

In [2]:
def print_entire_class(cls):
    source = inspect.getsource(cls)
    print(source)

print_entire_class(pya.models.Wu)

class Wu(LinearReferenceClock):
    def postprocess(self, x):
        """Horvath anti-log (adult age = 48) giving months, then months to years."""
        adult_age = 48
        mask_negative = x < 0
        mask_non_negative = ~mask_negative
        age = torch.empty_like(x)
        age[mask_negative] = (1 + adult_age) * torch.exp(x[mask_negative]) - 1
        age[mask_non_negative] = (1 + adult_age) * x[mask_non_negative] + adult_age
        return age / 12.0



In [3]:
model = pya.models.Wu()

## Define clock metadata

In [4]:
model.metadata["clock_name"] = 'wu'
model.metadata["data_type"] = 'methylation'
model.metadata["species"] = 'Homo sapiens'
model.metadata["year"] = 2019
model.metadata["approved_by_author"] = '⌛'
model.metadata["citation"] = "Wu, Xiaohui, et al. \"DNA methylation profile is a quantitative measure of biological aging in children.\" Aging 11.22 (2019): 10031-10051."
model.metadata["doi"] = "https://doi.org/10.18632/aging.102399"
model.metadata["research_only"] = None
model.metadata["notes"] = "Pediatric DNA-methylation age clock estimating chronological age (in months) from children's whole-blood methylation, built by sure independence screening followed by elastic-net regression over 111 CpGs, with methylation aging signatures largely distinct from adult clocks."
model.metadata["tissue"] = 'whole blood (peripheral blood leukocytes)'
model.metadata["predicts"] = 'chronological age'
model.metadata["unit"] = 'months'
model.metadata["model_type"] = 'Elastic net'
model.metadata["platform"] = 'Illumina 27K/450K'
model.metadata["population"] = 'pediatric/children (9-212 months, ~0-18 years)'
model.metadata["journal"] = 'Aging'
model.metadata["last_author"] = 'Huiying Liang'
model.metadata["n_features"] = 111
model.metadata["citations"] = 82
model.metadata["citations_date"] = '2026-07-05'

## Download clock dependencies

In [5]:
os.system(f"curl -sL -o wu.xlsx https://cdn.aging-us.com/article/102399/supplementary/SD3/0/aging-v11i22-102399-supplementary-material-SD3.xlsx")

0

## Load features

In [6]:
df = pd.read_excel('wu.xlsx', sheet_name='CpGs_information')
mask = df['CpGs'].astype(str).str.lower().isin(['intercept', '(intercept)'])
intercept_value = float(df.loc[mask, 'Active.coefficients'].iloc[0]) if mask.any() else 0.0
coef_df = df.loc[~mask].reset_index(drop=True)
model.features = coef_df['CpGs'].tolist()

## Load weights into base model

In [7]:
weights = torch.tensor(coef_df['Active.coefficients'].tolist()).unsqueeze(0).float()
intercept = torch.tensor([intercept_value]).float()

In [8]:
base_model = pya.models.LinearModel(input_dim=len(model.features))

base_model.linear.weight.data = weights.float()
base_model.linear.bias.data = intercept.float()

model.base_model = base_model

## Load reference values

In [9]:
model.reference_values = None

## Load preprocess and postprocess objects

In [10]:
model.preprocess_name = None
model.preprocess_dependencies = None

In [11]:
model.postprocess_name = 'anti_log_linear'
model.postprocess_dependencies = None

## Check all clock parameters

In [12]:
pya.utils.print_model_details(model)


%==================================== Model Details ====================================%
Model Attributes:

training: True
metadata: {'approved_by_author': '⌛',
 'citation': 'Wu, Xiaohui, et al. "DNA methylation profile is a quantitative '
             'measure of biological aging in children." Aging 11.22 (2019): '
             '10031-10051.',
 'clock_name': 'wu',
 'data_type': 'methylation',
 'doi': 'https://doi.org/10.18632/aging.102399',
 'notes': None,
 'research_only': None,
 'species': 'Homo sapiens',
 'version': None,
 'year': 2019}
reference_values: None
preprocess_name: None
preprocess_dependencies: None
postprocess_name: 'anti_log_linear'
postprocess_dependencies: None
features: ['cg00343092', 'cg00563932', 'cg00571634', 'cg00629217', 'cg01511567', 'cg01515426', 'cg01756060', 'cg01899253', 'cg02385474', 'cg02489552', 'cg02626929', 'cg02789485', 'cg03224418', 'cg03340261', 'cg03970609', 'cg04458548', 'cg04460372', 'cg04474832', 'cg04527989', 'cg04784672', 'cg05073035', 'cg0

## Basic test

In [13]:
torch.manual_seed(42)
input = torch.randn(10, len(model.features), dtype=float)
model.eval()
model.to(float)
pred = model(input)
pred

tensor([[ 7.1793e+01],
        [ 3.4194e+00],
        [-8.3333e-02],
        [ 2.7208e-01],
        [-8.3328e-02],
        [ 4.4315e+01],
        [ 1.1845e+02],
        [-8.2270e-02],
        [-8.3325e-02],
        [ 2.5722e+01]], dtype=torch.float64, grad_fn=<DivBackward0>)

## Save torch model

In [14]:
torch.save(model, f"../weights/{model.metadata['clock_name']}.pt")

## Clear directory
<a id="10"></a>

In [15]:
# Function to remove a folder and all its contents
def remove_folder(path):
    try:
        shutil.rmtree(path)
        print(f"Deleted folder: {path}")
    except Exception as e:
        print(f"Error deleting folder {path}: {e}")

# Get a list of all files and folders in the current directory
all_items = os.listdir('.')

# Loop through the items
for item in all_items:
    # Check if it's a file and does not end with .ipynb
    if os.path.isfile(item) and not item.endswith('.ipynb'):
        os.remove(item)
        print(f"Deleted file: {item}")
    # Check if it's a folder
    elif os.path.isdir(item):
        remove_folder(item)

Deleted file: wu.xlsx
